
# Exp.03 — PGR²M Entropy-Regularization Sensitivity

This experiment addresses **Reviewer 3, Concern 1**, which asks for a systematic analysis of
the entropy regularization used in the pose-guided RVQ tokenizer.

## Experimental design

The tokenizer objective in the released training code contains

\[
\mathcal{L} \leftarrow \mathcal{L}
+ \lambda_{\mathrm{RVQ}} \mathcal{L}_{\mathrm{RVQ}}
+ \lambda_{\mathrm{ent}} \mathcal{L}_{\mathrm{ent}},
\]

where `--params-soft-ent-loss` is \(\lambda_{\mathrm{ent}}\).

We evaluate:

- **0.000** — entropy regularization removed
- **0.005** — half of the default coefficient
- **0.010** — current/default coefficient (**reuse Exp.02 τ=0.1 matched run**)
- **0.020** — twice the default coefficient

All settings use the same architecture, seed, residual-dropout threshold \(\tau=0.1\),
learning rate, 300k training budget, and data pipeline.

### Why reuse λent=0.01?

The Exp.02 matched τ=0.1 run was already trained from scratch with the same seed and
all other hyperparameters fixed, including `--params-soft-ent-loss 0.01`.
Reusing it avoids approximately one additional 300k training run while preserving a
matched baseline.

### Expected cost

Only **three new 300k runs** are required. At the observed training speed, this is
approximately **28–32 hours total** on one A100, executed sequentially.

> This notebook only performs training + validation audit.  
> After completion, Exp.03B will evaluate each best-validation-FID checkpoint
> using the same 20-repeat HumanML3D test protocol used in Exp.02B.


In [ ]:

from pathlib import Path
import os, sys, json, re, time, hashlib, subprocess, shlex
import numpy as np
import pandas as pd

PGR = Path(".")
PY = Path("venv/bin/python")

# Reuse the known-good matched training wrapper from Exp.02.
WRAPPER = PGR / "revision_preflight" / "train_pg_tokenizer_exp02_seeded_compat.py"
LOSS_CFG = PGR / "configs" / "exp_dec" / "losses.yaml"

# Existing λ_ent=0.01 matched baseline (Exp.02 τ=0.1).
EXP02_ROOT = PGR / "exp_result_revision_tau_sensitivity"
EXP02_STATE = PGR / "revision_logs" / "exp02_tau" / "exp02_tau_queue_state.json"

# New Exp.03 outputs.
OUT_ROOT = PGR / "exp_result_revision_entropy_sensitivity"
LOGDIR = PGR / "revision_logs" / "exp03_entropy"
RESULTS = PGR / "revision_results" / "exp03_entropy"
PRE = PGR / "revision_preflight"

QUEUE_CONFIG = PRE / "exp03_entropy_queue_config.json"
QUEUE_RUNNER = PRE / "exp03_entropy_queue_runner.py"
QUEUE_STATE = LOGDIR / "exp03_entropy_queue_state.json"
QUEUE_PIDFILE = LOGDIR / "exp03_entropy_queue.pid"
QUEUE_MASTER_LOG = LOGDIR / "exp03_entropy_queue_master.log"

FREEZE = RESULTS / "exp03_entropy_config_freeze.json"
AUDIT_CSV = RESULTS / "exp03_entropy_validation_audit.csv"
TRAJ_CSV = RESULTS / "exp03_entropy_validation_trajectories.csv"

for d in [OUT_ROOT, LOGDIR, RESULTS, PRE]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print("=== EXP.03 ENVIRONMENT ===")
print("Python :", sys.executable)
print("Expected:", PY)
print("Wrapper:", WRAPPER)
print("Loss cfg:", LOSS_CFG)

assert PY.exists(), PY
assert WRAPPER.exists(), (
    f"Known-good Exp.02 training wrapper missing: {WRAPPER}\n"
    "Do not recreate it blindly; use the wrapper that successfully completed Exp.02."
)
assert LOSS_CFG.exists(), LOSS_CFG
assert EXP02_STATE.exists(), EXP02_STATE

if Path(sys.executable).resolve() != PY.resolve():
    print("[WARN] Notebook interpreter is not the Python 3 venv.")
    print("       Please select the Python 3 kernel before continuing.")

print("[OK] Hard gate passed.")


In [ ]:

# 2. Verify and freeze the reused λ_ent=0.01 baseline from Exp.02.

exp02_state = json.loads(EXP02_STATE.read_text())
rec = exp02_state.get("runs", {}).get("0.1", {})
assert rec.get("status") == "complete", f"Exp.02 τ=0.1 is not complete: {rec}"

baseline_log = Path(rec["log"])
assert baseline_log.exists(), baseline_log

baseline_base = EXP02_ROOT / "exp_pg_tokenizer_tau_0p1_seed123"
baseline_ckpts = sorted(
    baseline_base.rglob("net_best_fid.pth"),
    key=lambda p: p.stat().st_mtime
)
assert baseline_ckpts, f"No baseline net_best_fid.pth under {baseline_base}"

# Exp.02 should contain one completed production run; if multiple exist, print them.
if len(baseline_ckpts) > 1:
    print("[WARN] Multiple λ_ent=0.01 baseline candidates:")
    for p in baseline_ckpts:
        print("   ", p)

baseline_ckpt = baseline_ckpts[-1]
baseline_run_dir = baseline_ckpt.parent
baseline_args = baseline_run_dir / "arguments.yaml"
assert baseline_args.exists()

print("=== REUSED BASELINE ===")
print("λ_ent       : 0.01")
print("Run dir     :", baseline_run_dir)
print("Checkpoint  :", baseline_ckpt)
print("Checkpoint SHA256:", sha256(baseline_ckpt))
print("Log          :", baseline_log)

# Verify the baseline actually used entropy coefficient 0.01 and τ=0.1.
payload = json.loads(baseline_args.read_text())
assert abs(float(payload["params_soft_ent_loss"]) - 0.01) < 1e-12
assert abs(float(payload["pdrop_res"]) - 0.1) < 1e-12
assert int(payload["seed"]) == 123
assert int(payload["total_iter"]) == 300000

print("[OK] Baseline provenance verified.")


In [ ]:

# 3. Freeze the matched protocol.
#    Only λ_ent changes across conditions.

ENTROPY_WEIGHTS = [0.0, 0.005, 0.01, 0.02]
NEW_WEIGHTS = [0.0, 0.005, 0.02]

def ent_tag(x):
    if x == 0:
        return "0"
    s = f"{x:.3f}".rstrip("0").rstrip(".")
    return s.replace(".", "p")

COMMON = [
    "--batch-size", "256",
    "--lr", "2e-4",
    "--total-iter", "300000",
    "--warm-up-iter", "1000",
    "--eval-iter", "1000",
    "--print-iter", "200",
    "--lr-scheduler", "200000",
    "--nb-code", "392",
    "--down-t", "2",
    "--depth", "3",
    "--dilation-growth-rate", "3",
    "--dataname", "t2m",
    "--vq-act", "relu",
    "--loss-cfg-path", "./configs/exp_dec/losses.yaml",
    "--use-keywords",
    "--output-emb-width", "392",
    "--rvq-num-quantizers", "6",
    "--rvq-quantize-dropout-prob", "0.2",
    "--rvq-quantize-dropout-cutoff-index", "0",
    "--rvq-nb-code", "512",
    "--rvq-quantizer-type", "soft",
    "--rvq-vq-loss-beta", "0.25",
    "--rvq-loss-weight", "0.02",
    "--detach-p-latent",
    "--pdrop-res", "0.1",
    "--unuse-ema",
    "--num_workers", "4",
    "--seed", "123",
]

jobs = []
for w in NEW_WEIGHTS:
    log = LOGDIR / f"exp03_ent_{ent_tag(w)}.log"
    exp_name = f"exp_pg_tokenizer_ent_{ent_tag(w)}_seed123"
    cmd = [
        str(PY), "-u", str(WRAPPER),
        "--out-dir", str(OUT_ROOT),
        "--exp-name", exp_name,
        "--params-soft-ent-loss", str(w),
        *COMMON,
    ]
    jobs.append({
        "entropy_weight": w,
        "exp_name": exp_name,
        "log": str(log),
        "command": cmd,
    })

freeze = {
    "purpose": "Reviewer 3 entropy-regularization sensitivity",
    "entropy_weights": ENTROPY_WEIGHTS,
    "new_training_weights": NEW_WEIGHTS,
    "reused_baseline_weight": 0.01,
    "reused_baseline_run_dir": str(baseline_run_dir),
    "reused_baseline_checkpoint": str(baseline_ckpt),
    "reused_baseline_checkpoint_sha256": sha256(baseline_ckpt),
    "reused_baseline_log": str(baseline_log),
    "training_wrapper": str(WRAPPER),
    "training_wrapper_sha256": sha256(WRAPPER),
    "loss_cfg": str(LOSS_CFG),
    "loss_cfg_sha256": sha256(LOSS_CFG),
    "common_args": COMMON,
    "jobs": jobs,
}
FREEZE.write_text(json.dumps(freeze, indent=2))

print("=== EXP.03 FROZEN PROTOCOL ===")
display(pd.DataFrame([{
    "λ_ent": w,
    "source": "reuse Exp.02 τ=0.1" if w == 0.01 else "new 300k run",
} for w in ENTROPY_WEIGHTS]))
print("[OK] Freeze:", FREEZE)


In [ ]:

# 4. One-step production-batch smoke test at the strongest new coefficient (λ_ent=0.02).
#    This tests the exact wrapper, batch size, model architecture, and entropy coefficient.

SMOKE_OUT = PGR / "revision_preflight" / "exp03_entropy_smoke_out"
SMOKE_LOG = LOGDIR / "exp03_entropy_smoke_0p02.log"

smoke_cmd = [
    str(PY), "-u", str(WRAPPER),
    "--out-dir", str(SMOKE_OUT),
    "--exp-name", "exp03_entropy_smoke_0p02",
    "--params-soft-ent-loss", "0.02",
    "--batch-size", "256",
    "--lr", "2e-4",
    "--total-iter", "1",
    "--warm-up-iter", "2",
    "--eval-iter", "1",
    "--print-iter", "1",
    "--lr-scheduler", "200000",
    "--nb-code", "392",
    "--down-t", "2",
    "--depth", "3",
    "--dilation-growth-rate", "3",
    "--dataname", "t2m",
    "--vq-act", "relu",
    "--loss-cfg-path", "./configs/exp_dec/losses.yaml",
    "--use-keywords",
    "--output-emb-width", "392",
    "--rvq-num-quantizers", "6",
    "--rvq-quantize-dropout-prob", "0.2",
    "--rvq-quantize-dropout-cutoff-index", "0",
    "--rvq-nb-code", "512",
    "--rvq-quantizer-type", "soft",
    "--rvq-vq-loss-beta", "0.25",
    "--rvq-loss-weight", "0.02",
    "--detach-p-latent",
    "--pdrop-res", "0.1",
    "--unuse-ema",
    "--num_workers", "4",
    "--seed", "123",
]

print("=== EXP.03 SMOKE ===")
print(" ".join(shlex.quote(x) for x in smoke_cmd))

with open(SMOKE_LOG, "w") as f:
    p = subprocess.run(
        smoke_cmd,
        cwd=PGR,
        stdout=f,
        stderr=subprocess.STDOUT,
        timeout=7200,
    )

print("returncode:", p.returncode)
tail = SMOKE_LOG.read_text(errors="replace").splitlines()[-100:]
print("\n".join(tail))

if p.returncode != 0:
    raise RuntimeError(f"Exp.03 smoke failed. See {SMOKE_LOG}")

txt = SMOKE_LOG.read_text(errors="replace")
required = ["params_soft_ent_loss", "Train. Iter 1", "Eva. Iter 1"]
missing = [x for x in required if x not in txt]
if missing:
    raise RuntimeError(f"Smoke returned 0 but expected markers are missing: {missing}")

print("[OK] Production-batch entropy smoke passed.")


In [ ]:

# 5. Build the detached, resumable, sequential queue for the THREE new 300k runs.

QUEUE_CONFIG.write_text(json.dumps({
    "state_path": str(QUEUE_STATE),
    "master_log": str(QUEUE_MASTER_LOG),
    "jobs": jobs,
}, indent=2))

runner = r'''
import json, re, subprocess, sys, time
from pathlib import Path

cfg = json.loads(Path(sys.argv[1]).read_text())
state_path = Path(cfg["state_path"])
master_log = Path(cfg["master_log"])
jobs = cfg["jobs"]

train_iter_pat = re.compile(r"Train\. Iter\s+(\d+)\s*:")

def max_iter_from_log(path):
    path = Path(path)
    if not path.exists():
        return None
    m = None
    for line in path.read_text(errors="replace").splitlines():
        z = train_iter_pat.search(line)
        if z:
            m = int(z.group(1))
    return m

def has_checkpoint(exp_name):
    base = Path("exp_result_revision_entropy_sensitivity") / exp_name
    return any(base.rglob("net_best_fid.pth")) if base.exists() else False

def load_state():
    if state_path.exists():
        return json.loads(state_path.read_text())
    return {
        "queue_status": "running",
        "runs": {
            str(j["entropy_weight"]): {
                "status": "pending",
                "log": j["log"],
                "exp_name": j["exp_name"],
            } for j in jobs
        }
    }

def save(s):
    state_path.write_text(json.dumps(s, indent=2))

def master(msg):
    with master_log.open("a") as f:
        f.write(msg.rstrip() + "\n")
        f.flush()

state = load_state()
state["queue_status"] = "running"
save(state)

for j in jobs:
    k = str(j["entropy_weight"])
    rec = state["runs"].setdefault(k, {})
    log_path = Path(j["log"])

    # Re-launch safety: skip already-complete production runs.
    if rec.get("status") == "complete":
        mi = max_iter_from_log(log_path)
        ck = has_checkpoint(j["exp_name"])
        if mi == 300000 and ck:
            master(f"[SKIP] lambda_ent={j['entropy_weight']} already complete")
            continue

    rec.update({
        "status": "running",
        "log": j["log"],
        "exp_name": j["exp_name"],
        "started_at": time.time(),
    })
    save(state)

    master("[START] lambda_ent={} {}".format(
        j["entropy_weight"], " ".join(j["command"])
    ))

    with log_path.open("w") as f:
        p = subprocess.run(
            j["command"],
            cwd=".",
            stdout=f,
            stderr=subprocess.STDOUT,
        )

    mi = max_iter_from_log(log_path)
    ck = has_checkpoint(j["exp_name"])
    ok = (p.returncode == 0 and mi == 300000 and ck)

    rec.update({
        "returncode": p.returncode,
        "max_iter": mi,
        "checkpoint": ck,
        "finished_at": time.time(),
        "status": "complete" if ok else "failed",
    })
    save(state)

    master(
        f"[END] lambda_ent={j['entropy_weight']} rc={p.returncode} "
        f"max_iter={mi} checkpoint={ck} status={rec['status']}"
    )

    if not ok:
        state["queue_status"] = "failed"
        save(state)
        master("[QUEUE FAILED]")
        sys.exit(1)

state["queue_status"] = "complete"
save(state)
master("[QUEUE COMPLETE] all new entropy runs finished")
'''

QUEUE_RUNNER.write_text(runner)
compile(QUEUE_RUNNER.read_text(), str(QUEUE_RUNNER), "exec")

print("[OK] Queue config:", QUEUE_CONFIG)
print("[OK] Queue runner:", QUEUE_RUNNER)
display(pd.DataFrame(jobs)[["entropy_weight", "exp_name", "log"]])


In [ ]:

# 6. Launch Exp.03 queue.
#    Safe PID guard: do not start a second queue if one is already running.

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

old_pid = None
if QUEUE_PIDFILE.exists():
    try:
        old_pid = int(QUEUE_PIDFILE.read_text().strip())
    except Exception:
        pass

if old_pid and pid_alive(old_pid):
    print("=== EXP.03 QUEUE ALREADY RUNNING ===")
    print("PID  :", old_pid)
    print("Alive:", True)
else:
    master_handle = open(QUEUE_MASTER_LOG, "a")
    p = subprocess.Popen(
        [str(PY), "-u", str(QUEUE_RUNNER), str(QUEUE_CONFIG)],
        cwd=PGR,
        stdout=master_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    QUEUE_PIDFILE.write_text(str(p.pid))
    time.sleep(1)

    print("=== EXP.03 QUEUE LAUNCH ===")
    print("PID   :", p.pid)
    print("Alive :", pid_alive(p.pid))
    print("State :", QUEUE_STATE)
    print("Log   :", QUEUE_MASTER_LOG)


In [ ]:

# 7. Read-only status cell — safe to run at any time during training.

print("=== EXP.03 QUEUE STATUS ===")

pid = None
if QUEUE_PIDFILE.exists():
    try:
        pid = int(QUEUE_PIDFILE.read_text().strip())
    except Exception:
        pass

print("PID   :", pid)
print("Alive :", pid_alive(pid) if pid else False)

state = json.loads(QUEUE_STATE.read_text()) if QUEUE_STATE.exists() else {}
print("Queue status:", state.get("queue_status", "not_started"))

iter_pat = re.compile(r"Train\. Iter\s+(\d+)\s*:")

def current_iter(log_path):
    p = Path(log_path)
    if not p.exists():
        return None
    last = None
    for line in p.read_text(errors="replace").splitlines():
        m = iter_pat.search(line)
        if m:
            last = int(m.group(1))
    return last

for w in NEW_WEIGHTS:
    rec = state.get("runs", {}).get(str(w), {})
    status = rec.get("status", "pending")
    log = rec.get("log")
    it = current_iter(log) if log else None
    pct = (100.0 * it / 300000) if it is not None else 0.0
    print(
        f"λ_ent={w}: status={status} iter={it} ({pct:.2f}%) "
        f"log={log}"
    )

print("λ_ent=0.01: reused completed Exp.02 τ=0.1 baseline")

print("\n=== MASTER LOG TAIL ===")
if QUEUE_MASTER_LOG.exists():
    print("\n".join(QUEUE_MASTER_LOG.read_text(errors="replace").splitlines()[-30:]))


In [ ]:

# 8. Post-run validation audit.
#    Run only when the three new runs are complete.
#
#    Metrics:
#      - best-validation-FID checkpoint iteration
#      - validation FID at that iteration
#      - validation residual-codebook perplexity at that iteration
#      - final 300k FID / perplexity
#
#    The λ_ent=0.01 row is read from the already completed Exp.02 baseline.

if not QUEUE_STATE.exists():
    raise RuntimeError("No Exp.03 queue state exists.")

state = json.loads(QUEUE_STATE.read_text())
if state.get("queue_status") != "complete":
    raise RuntimeError(
        f"Queue is not complete yet: {state.get('queue_status')}. "
        "Re-run the status cell later."
    )

# Discover run directories/checkpoints.
condition_info = {
    0.01: {
        "run_dir": baseline_run_dir,
        "checkpoint": baseline_ckpt,
        "log": baseline_log,
        "source": "Exp.02 τ=0.1 reused baseline",
    }
}

for w in NEW_WEIGHTS:
    exp_name = f"exp_pg_tokenizer_ent_{ent_tag(w)}_seed123"
    base = OUT_ROOT / exp_name
    ckpts = sorted(base.rglob("net_best_fid.pth"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise RuntimeError(f"No net_best_fid.pth found for λ_ent={w}: {base}")
    ck = ckpts[-1]
    condition_info[w] = {
        "run_dir": ck.parent,
        "checkpoint": ck,
        "log": Path(state["runs"][str(w)]["log"]),
        "source": "Exp.03 new matched run",
    }

fid_pat = re.compile(r"Eva\. Iter\s+(\d+)\s+:,?\s*FID\.\s*([-+0-9.eE]+)")
perp_pat = re.compile(r"Validation\. Iter\s+(\d+)\s*:\s*Perplexity\.\s*([-+0-9.eE]+)")

def parse_trajectory(log_path):
    lines = Path(log_path).read_text(errors="replace").splitlines()

    fids = {}
    perps = {}

    # There can be multiple Eva lines per iteration (pose-only then full residual).
    # The full-residual validation is logged last at each iteration, so keep the last one.
    for line in lines:
        m = fid_pat.search(line)
        if m and line[:4].isdigit():  # timestamped line only; avoid duplicate logger echo
            fids[int(m.group(1))] = float(m.group(2))

        m = perp_pat.search(line)
        if m and line[:4].isdigit():
            perps[int(m.group(1))] = float(m.group(2))

    return fids, perps

audit_rows = []
traj_rows = []

for w in ENTROPY_WEIGHTS:
    info = condition_info[w]
    ckpt = info["checkpoint"]

    # The production checkpoints used in the earlier experiments contain nb_iter.
    import torch
    ck = torch.load(ckpt, map_location="cpu")
    best_iter = ck.get("nb_iter")
    if best_iter is None:
        raise RuntimeError(
            f"Checkpoint for λ_ent={w} has no nb_iter metadata: {ckpt}"
        )
    best_iter = int(best_iter)

    fids, perps = parse_trajectory(info["log"])

    if best_iter not in fids:
        raise RuntimeError(
            f"No full validation FID found at best checkpoint iter={best_iter} "
            f"for λ_ent={w}. Inspect {info['log']}"
        )

    best_fid = fids[best_iter]
    best_perp = perps.get(best_iter, np.nan)
    final_fid = fids.get(300000, np.nan)
    final_perp = perps.get(300000, np.nan)

    audit_rows.append({
        "entropy_weight": w,
        "source": info["source"],
        "best_fid_checkpoint_iter": best_iter,
        "best_val_fid": best_fid,
        "val_perplexity_at_best": best_perp,
        "final_fid_300k": final_fid,
        "final_perplexity_300k": final_perp,
        "checkpoint": str(ckpt),
        "checkpoint_sha256": sha256(ckpt),
        "log": str(info["log"]),
    })

    all_iters = sorted(set(fids) | set(perps))
    for it in all_iters:
        traj_rows.append({
            "entropy_weight": w,
            "iter": it,
            "fid": fids.get(it, np.nan),
            "perplexity": perps.get(it, np.nan),
        })

audit = pd.DataFrame(audit_rows).sort_values("entropy_weight")
traj = pd.DataFrame(traj_rows).sort_values(["entropy_weight", "iter"])

audit.to_csv(AUDIT_CSV, index=False)
traj.to_csv(TRAJ_CSV, index=False)

print("=== EXP.03 VALIDATION AUDIT ===")
display(audit[[
    "entropy_weight",
    "best_fid_checkpoint_iter",
    "best_val_fid",
    "val_perplexity_at_best",
    "final_fid_300k",
    "final_perplexity_300k",
    "source",
]])

print("Saved:")
print(" ", AUDIT_CSV)
print(" ", TRAJ_CSV)


In [ ]:

# 9. Final decision.

assert AUDIT_CSV.exists(), AUDIT_CSV
audit = pd.read_csv(AUDIT_CSV)

ok_weights = np.allclose(
    sorted(audit["entropy_weight"].astype(float).tolist()),
    ENTROPY_WEIGHTS,
)
ok_ckpt = all(Path(p).exists() for p in audit["checkpoint"])
ok_finite = np.isfinite(
    audit[[
        "best_val_fid",
        "val_perplexity_at_best",
        "final_fid_300k",
        "final_perplexity_300k",
    ]].to_numpy(dtype=float)
).all()

print("=== EXP.03 FINAL STATUS ===")
print("Entropy weights present   :", ok_weights)
print("Best-FID checkpoints ready:", ok_ckpt)
print("Audit metrics finite      :", ok_finite)
print("Validation audit          :", AUDIT_CSV)

FINALIZED = bool(ok_weights and ok_ckpt and ok_finite)
print("FINALIZED                 :", FINALIZED)

if not FINALIZED:
    raise RuntimeError("Exp.03 validation stage is not finalized.")

print("\nDECISION: EXP.03 TRAINING IS COMPLETE.")
print("Next: Exp.03B — 20-repeat HumanML3D test evaluation of the four best-FID checkpoints.")
